# 05 — Fig. 5 local-conformal landscape, corrected fixed seed 43

This notebook is the sole source of the repository Fig. 5. It renders a fixed split directly to `results/figures/fig5.png`.

## Fixed configuration

- Registry split: `split_01`, seed `43`.
- Point-model outputs: stored corrected `Plain` model calibration and test predictions; CatBoost is not retrained.
- Historical article visual template: 2×2 layout, arrows, palette, contours and axis limits.
- Fixed historical colour limits: panel a 2.6–4.0; panel b 0.2086297262–0.7389536538; panel c upper limit 1.85; panel d 88–92%.

Panels a–b are descriptive whole-cohort surfaces retained for visual continuity with the historical Fig. 5. Panel c uses calibration residuals. Panel d uses held-out test coverage for the fixed split. The figure is illustrative; the 30-split audit remains the aggregate evidence.

In [ ]:
from __future__ import annotations

import hashlib
import json
import math
import platform
import sys
from datetime import date
from pathlib import Path

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import matplotlib.patheffects as pe
from matplotlib.colors import LinearSegmentedColormap, to_rgba
import numpy as np
import pandas as pd
from PIL import Image

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
DATA_PATH = ROOT / "data" / "df_final.csv"
SPLIT_PATH = ROOT / "data" / "splits" / "split_indices.csv"
MODEL_ROOT = ROOT / "results" / "models"
FIGURE_PATH = ROOT / "results" / "figures" / "fig5.png"
TABLE_DIR = ROOT / "results" / "tables"
FIGURE_PATH.parent.mkdir(parents=True, exist_ok=True)

TARGET = "-lgLD50, mol/kg"
FIXED_SPLIT_INDEX = 1
FIXED_SEED = 43
PROTEINS = [
    "1m2z", "1pbq", "1xoq", "2rh1", "2vt4", "2ydo", "2z5x", "3b66", "3kk6", "3ln1",
    "3rze", "4djh", "4ey7", "4iar", "4mqs", "4n6h", "5cxv", "5i71", "5tvn", "5u09",
    "5va1", "6cm4", "6kpf", "6kux", "6lqa", "6pdj", "6x3x", "6y1z", "7f8y", "7kwe",
    "7ljd", "7wc9", "7xnk", "7ym8", "8e9y", "8ef6", "8fhs", "8pjk", "8st0", "8wty",
    "8xvk", "8yn3", "9eo4", "V1A",
]
BTOX_COLUMNS = ["hERG", "Respiratory", "Neurotoxicity-DI"]
ALPHA = 0.10
H_CONFORMAL = 0.90
NEFF_MIN_CONFORMAL = 80
H_MAP = 0.45
H_COVERAGE = 0.75
NEFF_MIN_MAP = 60
DISPLAY_NEFF_MIN = 120
GRID = 80
X_RANGE = (-9, -4)
COV_VMIN = 88.0
COV_VMAX = 92.0
MEAN_VMIN = 2.6
MEAN_VMAX = 4.0
VARIANCE_VMIN = 0.20862972624627632
VARIANCE_VMAX = 0.7389536537902531

INK = "#16222e"
GRAPHITE = "#5a6b7a"
SUBTLE = "#7d8a97"
GRID_C = "#e6ebf0"
LOW_SUPPORT = "#6f7a83"
SOFT_CONTOUR_LIGHT = "#7f929e"
SOFT_CONTOUR_DARK = "#f1f5f8"
ARROW_COLOR = "#ffffff"
ARROW_SHADOW = "#000000"
CMAP = LinearSegmentedColormap.from_list(
    "blue_graphite", ["#f4f7fa", "#dbe6ee", "#b3cbdd", "#7ba3c3", "#3f6f97", "#16313f"], N=256
)
CMAP.set_bad(LOW_SUPPORT)


def sha256(path: Path) -> str:
    digest = hashlib.sha256()
    with path.open("rb") as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()

for required in [DATA_PATH, SPLIT_PATH]:
    assert required.is_file(), required
print("Repository root: current working copy")
print(f"Fixed Fig. 5: split_{FIXED_SPLIT_INDEX:02d}, seed={FIXED_SEED}")

In [ ]:
def weighted_quantiles_and_neff(query_features, calibration_features, calibration_residuals, quantile_level, bandwidth, batch_size=256):
    order = np.argsort(calibration_residuals)
    sorted_residuals = calibration_residuals[order]
    quantiles = np.empty(len(query_features), dtype=float)
    neff = np.empty(len(query_features), dtype=float)
    for start in range(0, len(query_features), batch_size):
        end = min(start + batch_size, len(query_features))
        difference = query_features[start:end, None, :] - calibration_features[None, :, :]
        weights = np.exp(-0.5 * np.sum(difference * difference, axis=2) / (bandwidth**2))
        total_weight = weights.sum(axis=1)
        neff[start:end] = total_weight**2 / np.sum(weights * weights, axis=1)
        cumulative = np.cumsum(weights[:, order], axis=1) / total_weight[:, None]
        quantiles[start:end] = sorted_residuals[np.argmax(cumulative >= quantile_level, axis=1)]
    return quantiles, neff


def standardize(calibration_raw, values_raw):
    location = np.nanmean(calibration_raw, axis=0)
    scale = np.nanstd(calibration_raw, axis=0)
    scale[~np.isfinite(scale) | (scale == 0)] = 1.0
    return (calibration_raw - location) / scale, (values_raw - location) / scale


def make_grid(x, y):
    xs = np.linspace(np.percentile(x, 0.5), np.percentile(x, 99.5), GRID)
    ys = np.linspace(np.percentile(y, 0.5), np.percentile(y, 99.5), GRID)
    return xs, ys, float(np.std(x)), float(np.std(y))


def nw_mean_var_neff(x, y, target, xs, ys, sx, sy):
    xz, yz = x / sx, y / sy
    mean = np.full((len(ys), len(xs)), np.nan)
    variance = np.full_like(mean, np.nan)
    neff = np.full_like(mean, np.nan)
    for row, gy in enumerate(ys):
        dy = gy / sy - yz
        for col, gx in enumerate(xs):
            dx = gx / sx - xz
            weights = np.exp(-0.5 * (dx * dx + dy * dy) / (H_MAP**2))
            total_weight = weights.sum()
            effective_n = total_weight**2 / np.sum(weights * weights)
            neff[row, col] = effective_n
            if effective_n < NEFF_MIN_MAP:
                continue
            local_mean = np.sum(weights * target) / total_weight
            mean[row, col] = local_mean
            variance[row, col] = np.sum(weights * (target - local_mean)**2) / total_weight
    return mean, variance, neff


def nw_surface(x, y, values, xs, ys, sx, sy, bandwidth):
    xz, yz = x / sx, y / sy
    out = np.full((len(ys), len(xs)), np.nan)
    neff = np.full_like(out, np.nan)
    for row, gy in enumerate(ys):
        dy = gy / sy - yz
        for col, gx in enumerate(xs):
            dx = gx / sx - xz
            weights = np.exp(-0.5 * (dx * dx + dy * dy) / (bandwidth**2))
            total_weight = weights.sum()
            effective_n = total_weight**2 / np.sum(weights * weights)
            neff[row, col] = effective_n
            if effective_n < NEFF_MIN_MAP:
                continue
            out[row, col] = np.sum(weights * values) / total_weight
    return out, neff


def add_panel_label(ax, label):
    text = ax.text(-0.018, 0.985, label, transform=ax.transAxes, color=INK, fontsize=14, fontweight="bold", ha="right", va="top", zorder=12, clip_on=False)
    text.set_path_effects([pe.Stroke(linewidth=2.2, foreground="white", alpha=0.95), pe.Normal()])


def draw_soft_contours(ax, zmat, extent, fmt="%.2f"):
    valid = zmat[np.isfinite(zmat)]
    if len(valid) < 2:
        return
    levels = np.linspace(np.nanpercentile(valid, 4), np.nanpercentile(valid, 96), 8)
    midpoint = len(levels) // 2
    for subset, color in [(levels[:midpoint], SOFT_CONTOUR_LIGHT), (levels[midpoint:], SOFT_CONTOUR_DARK)]:
        if len(subset):
            cs = ax.contour(zmat, extent=extent, origin="lower", levels=subset, colors=[to_rgba(color, 0.72)], linewidths=0.62)
            ax.clabel(cs, fmt=fmt, fontsize=7, colors=color, inline_spacing=3, inline=True)


def draw_coverage_contours(ax, zmat, extent):
    valid = zmat[np.isfinite(zmat)]
    if len(valid) < 2:
        return
    for level, style, width, color, alpha in [(89, "dashed", 0.45, "#b0b0b0", 0.60), (90, "solid", 0.80, "#ffffff", 1.00), (91, "dashed", 0.45, "#b0b0b0", 0.60)]:
        if np.nanmin(valid) < level < np.nanmax(valid):
            cs = ax.contour(zmat, extent=extent, origin="lower", levels=[level], colors=[color], linewidths=width, linestyles=style, alpha=alpha)
            ax.clabel(cs, fmt="%.0f%%", fontsize=6.5, colors=color, inline_spacing=2, inline=True)


def add_explanatory_arrows(ax):
    def arrow(start, end, width=1.8, alpha=0.80):
        annotation = ax.annotate("", xy=end, xytext=start, xycoords="axes fraction", textcoords="axes fraction", arrowprops=dict(arrowstyle="-|>", color=ARROW_COLOR, alpha=alpha, linewidth=width, mutation_scale=21, shrinkA=0, shrinkB=0), zorder=8)
        annotation.arrow_patch.set_path_effects([pe.Stroke(linewidth=width + 2.0, foreground=ARROW_SHADOW, alpha=0.30), pe.Normal()])
    def label(text, x, y, rotation=0, size=11.25):
        artist = ax.text(x, y, text, transform=ax.transAxes, color=ARROW_COLOR, fontsize=size, rotation=rotation, rotation_mode="anchor", ha="center", va="center", zorder=9)
        artist.set_path_effects([pe.Stroke(linewidth=0.8, foreground=ARROW_SHADOW), pe.Normal()])
    arrow((0.86, 0.055), (0.055, 0.055))
    label("stronger panel-wide protein–ligand binding", 0.49, 0.10, size=11.1)
    arrow((0.045, 0.16), (0.045, 0.88))
    label("higher predicted organ toxicity", 0.09, 0.52, rotation=90)
    arrow((0.83, 0.18), (0.20, 0.84), width=1.7, alpha=0.78)
    diagonal_angle = np.degrees(np.arctan2(0.84 - 0.18, 0.20 - 0.83)) - 180.0
    label("higher in vivo acute toxicity", 0.55, 0.54, rotation=diagonal_angle, size=10.9)


def add_diagonal_arrow(ax, text):
    start, end = (0.84, 0.16), (0.18, 0.82)
    angle = np.degrees(np.arctan2(end[1] - start[1], end[0] - start[0])) - 180.0
    ann = ax.annotate("", xy=end, xytext=start, xycoords="axes fraction", textcoords="axes fraction", arrowprops=dict(arrowstyle="-|>", color=ARROW_COLOR, alpha=0.78, linewidth=1.7, mutation_scale=20, shrinkA=0, shrinkB=0), zorder=8)
    ann.arrow_patch.set_path_effects([pe.Stroke(linewidth=3.6, foreground=ARROW_SHADOW, alpha=0.30), pe.Normal()])
    artist = ax.text(0.51, 0.55, text, transform=ax.transAxes, color=ARROW_COLOR, fontsize=11.0, rotation=angle, rotation_mode="anchor", ha="center", va="center", zorder=9)
    artist.set_path_effects([pe.Stroke(linewidth=0.8, foreground=ARROW_SHADOW), pe.Normal()])


def crop_top_whitespace(path):
    with Image.open(path) as image:
        array = np.asarray(image.convert("RGB"))
        start = 0
        for row in range(array.shape[0]):
            if np.sum(np.any(array[row] < 240, axis=1)) > 50:
                start = row
                break
        if start:
            image.crop((0, start, image.width, image.height)).save(path)
    return start


def visual_distance(candidate_path, legacy_path, size=(512, 512)):
    with Image.open(candidate_path) as candidate, Image.open(legacy_path) as legacy:
        candidate_array = np.asarray(candidate.convert("RGB").resize(size), dtype=np.float32) / 255.0
        legacy_array = np.asarray(legacy.convert("RGB").resize(size), dtype=np.float32) / 255.0
    difference = candidate_array - legacy_array
    return float(np.mean(np.abs(difference))), float(np.sqrt(np.mean(difference * difference)))

In [ ]:
use_columns = ["ligand_id", TARGET, *PROTEINS, *BTOX_COLUMNS]
df = pd.read_csv(DATA_PATH, usecols=use_columns)
assert df["ligand_id"].is_unique
energies = df[PROTEINS].apply(pd.to_numeric, errors="coerce").to_numpy(float)
df["Emean"] = np.nanmean(energies, axis=1)
df[BTOX_COLUMNS] = df[BTOX_COLUMNS].apply(pd.to_numeric, errors="coerce").fillna(0.0).clip(0.0, 1.0)
df["B_Tox"] = df[BTOX_COLUMNS].sum(axis=1)
by_id = df.set_index("ligand_id", drop=False)
registry = pd.read_csv(SPLIT_PATH)
split_indices = sorted(registry["split_index"].unique())
assert split_indices == list(range(30)), split_indices

# Historical-style descriptive surfaces, intentionally shared by every split.
x_all = df["Emean"].to_numpy(float)
y_all_coordinate = df["B_Tox"].to_numpy(float)
y_all = df[TARGET].to_numpy(float)
xs, ys, sx, sy = make_grid(x_all, y_all_coordinate)
mean_map, variance_map, map_neff = nw_mean_var_neff(x_all, y_all_coordinate, y_all, xs, ys, sx, sy)
support_mask = map_neff >= DISPLAY_NEFF_MIN
mean_display = np.where(support_mask, mean_map, np.nan)
variance_display = np.where(support_mask, variance_map, np.nan)
variance_valid = variance_display[np.isfinite(variance_display)]
extent = [xs[0], xs[-1], ys[0], ys[-1]]

all_raw_features = df[["Emean", *BTOX_COLUMNS]].to_numpy(float)
row_for_id = pd.Series(df.index.to_numpy(), index=df["ligand_id"].astype(int))
print('Corrected molecules:', len(df), 'registered splits:', len(split_indices))

In [ ]:
def render_fixed_split():
    split_index = FIXED_SPLIT_INDEX
    seed = FIXED_SEED
    model_dir = MODEL_ROOT / f"split_{split_index:02d}" / "plain"
    calibration_path = model_dir / "calibration_predictions.csv"
    test_path = model_dir / "test_predictions_full_test.csv"
    conformal_path = model_dir / "conformal.json"
    for item in [calibration_path, test_path, conformal_path]:
        assert item.is_file(), item
    calibration = pd.read_csv(calibration_path)
    test = pd.read_csv(test_path)
    conformal = json.loads(conformal_path.read_text())
    calibration_ids = calibration["ligand_id"].astype(int).to_numpy()
    test_ids = test["ligand_id"].astype(int).to_numpy()
    expected_calibration = registry.loc[(registry["split_index"] == split_index) & (registry["set"] == "val"), "ligand_id"].astype(int).to_numpy()
    expected_test = registry.loc[(registry["split_index"] == split_index) & (registry["set"] == "test"), "ligand_id"].astype(int).to_numpy()
    assert set(calibration_ids) == set(expected_calibration)
    assert set(test_ids) == set(expected_test)

    calibration_raw = by_id.loc[calibration_ids, ["Emean", *BTOX_COLUMNS]].to_numpy(float)
    calibration_standardized, all_standardized = standardize(calibration_raw, all_raw_features)
    residuals = calibration["abs_residual"].to_numpy(float)
    quantile_level = min(1.0, math.ceil((len(residuals) + 1) * (1 - ALPHA)) / len(residuals))
    global_half_width = float(conformal["half_width_quantiles"]["CI_0.90"])
    local_quantiles, local_neff = weighted_quantiles_and_neff(
        all_standardized, calibration_standardized, residuals, quantile_level, H_CONFORMAL
    )
    local_width_all = 2.0 * np.where(local_neff < NEFF_MIN_CONFORMAL, global_half_width, local_quantiles)
    width_map, _ = nw_surface(x_all, y_all_coordinate, local_width_all, xs, ys, sx, sy, H_MAP)
    width_display = np.where(support_mask, width_map, np.nan)

    test_rows = row_for_id.loc[test_ids].to_numpy(int)
    test_width = local_width_all[test_rows]
    test_half_width = test_width / 2.0
    test_covered = ((test["y_true"].to_numpy(float) >= test["y_pred"].to_numpy(float) - test_half_width) &
                    (test["y_true"].to_numpy(float) <= test["y_pred"].to_numpy(float) + test_half_width)).astype(float)
    coverage_map, coverage_neff = nw_surface(
        x_all[test_rows], y_all_coordinate[test_rows], test_covered, xs, ys, sx, sy, H_COVERAGE
    )
    coverage_display = np.where(support_mask & (coverage_neff >= NEFF_MIN_MAP), coverage_map * 100.0, np.nan)

    fig, axes = plt.subplots(2, 2, figsize=(10.5, 10.2), facecolor="white", sharex="all", sharey="all")
    panel_definitions = [
        # Fixed to the final article Fig. 5 scale: 2.6 to 4.0 in 0.2 increments.
        (mean_display, "Mean local −lgLD50", float(y_all.mean()), "global μ", 2.6, 4.0, "Mean −lgLD50, mol/kg"),
        # Fixed to the final article Fig. 5 variance scale.
        (variance_display, "Variance of local −lgLD50", float(y_all.var()), "global σ²", 0.20862972624627632, 0.7389536537902531, "Variance −lgLD50, (mol/kg)²"),
        (width_display, "90% conformal interval width\nNW-Conformal Predictor over CatBoost plain", float(np.nanmean(local_width_all)), "all mean width", None, 1.85, "Interval width, −lgLD50, mol/kg"),
        (coverage_display, "Smoothed empirical coverage\nNW-Conformal Predictor over CatBoost plain", float(test_covered.mean() * 100), "overall empirical coverage", COV_VMIN, COV_VMAX, "Observed coverage, %"),
    ]
    for panel, (ax, definition) in enumerate(zip(axes.flat, panel_definitions)):
        zmat, title, statistic, statistic_label, vmin, vmax, cbar_label = definition
        add_panel_label(ax, "abcd"[panel])
        image = ax.imshow(np.ma.masked_invalid(zmat), extent=extent, origin="lower", aspect="auto", cmap=CMAP, interpolation="bilinear", vmin=vmin, vmax=vmax)
        ax.set_box_aspect(1)
        if panel == 0:
            draw_soft_contours(ax, zmat, extent)
            add_explanatory_arrows(ax)
        elif panel == 1:
            draw_soft_contours(ax, zmat, extent)
            add_diagonal_arrow(ax, "lower local data variance")
        elif panel == 2:
            draw_soft_contours(ax, zmat, extent)
            add_diagonal_arrow(ax, "narrower interval = lower uncertainty")
        else:
            draw_coverage_contours(ax, zmat, extent)
        colorbar = fig.colorbar(image, ax=ax, fraction=0.046, pad=0.04)
        if panel == 0:
            colorbar.set_ticks(np.round(np.arange(2.6, 4.01, 0.2), 1))
        elif panel == 2:
            colorbar.set_ticks(np.round(np.arange(1.1, 1.81, 0.1), 1))
        elif panel == 3:
            colorbar.set_ticks(np.arange(88, 93, 1))
        colorbar.set_label(cbar_label, fontsize=8.5, color=INK)
        colorbar.ax.tick_params(colors=INK, labelsize=7)
        ax.set_xlim(X_RANGE)
        ax.tick_params(colors=INK, labelsize=7)
        ax.set_title(title, fontsize=10, fontweight="normal", pad=8, color=INK)
        suffix = "%" if panel == 3 else ""
        ax.text(0.09, 0.98, f"{statistic_label} = {statistic:.2f}{suffix}", transform=ax.transAxes, fontsize=7.5, color=SUBTLE, va="top", ha="left", zorder=10, bbox=dict(boxstyle="round,pad=0.25", facecolor="white", alpha=0.75, edgecolor=GRID_C))
    for ax in axes[0, :]:
        ax.set_xlabel("")
    for ax in axes[1, :]:
        ax.set_xlabel("MIE proxy — mean binding across 44 targets\n(mean E, kcal/mol)", fontsize=9, color=INK)
    for ax in axes[:, 1]:
        ax.set_ylabel("")
    for ax in axes[:, 0]:
        ax.set_ylabel("Toxicity-phenotype burden\nhERG + Resp. + Neurotox.", fontsize=9, color=INK)
    axes[0, 0].tick_params(labelbottom=True)
    axes[0, 1].tick_params(labelbottom=True, labelleft=True)
    axes[1, 1].tick_params(labelleft=True)
    fig.tight_layout(rect=[0, 0, 1, 1])
    fig.subplots_adjust(hspace=0.05)
    figure_path = FIGURE_PATH
    fig.savefig(figure_path, dpi=220, bbox_inches="tight", facecolor="white")
    plt.close(fig)
    crop_rows = crop_top_whitespace(figure_path)
    with Image.open(figure_path) as image:
        image.verify()
    return {
        "split_index": int(split_index), "seed": seed, "figure": str(figure_path.relative_to(ROOT)),
        "n_calibration": int(len(calibration)), "n_test": int(len(test)),
        "global_interval_width_90": float(2 * global_half_width),
        "local_test_mean_width_90": float(test_width.mean()),
        "local_test_coverage_90": float(test_covered.mean()),
        "local_test_fallback_fraction": float((local_neff[test_rows] < NEFF_MIN_CONFORMAL).mean()),
        "grid_fraction_displayed": float(np.isfinite(coverage_display).mean()),
        "top_crop_rows": int(crop_rows),
        "input_calibration_sha256": sha256(calibration_path),
        "input_test_predictions_sha256": sha256(test_path),
        "input_conformal_sha256": sha256(conformal_path),
    }

summary = pd.DataFrame([render_fixed_split()])
assert int(summary.iloc[0]["split_index"]) == FIXED_SPLIT_INDEX
assert int(summary.iloc[0]["seed"]) == FIXED_SEED
summary_path = TABLE_DIR / "fig5_summary.csv"
summary.to_csv(summary_path, index=False)
print(summary.to_string(index=False))

In [ ]:
# The caption and detailed provenance stay with the article sources.  This package writes only the final figure and its compact CSV.
print("Final Figure 5 outputs:", FIGURE_PATH.relative_to(ROOT), summary_path.relative_to(ROOT))


In [ ]:
required = [FIGURE_PATH, summary_path]
assert all(path.is_file() and path.stat().st_size > 0 for path in required)
with Image.open(FIGURE_PATH) as image:
    image.verify()
with Image.open(FIGURE_PATH) as image:
    assert image.width >= 1500 and image.height >= 1500
print("Fig. 5 fixed seed-43 final outputs: PASS")


## Interpretation boundary

Fig. 5 is a reproducible illustration of the fixed registered `seed=43` split. It preserves the historical article visual configuration while using corrected data and stored corrected model outputs. The figure does not replace the repeated-split audit or imply local conditional calibration.

In [ ]:
# Artifact preview — show this stage's fixed-split figure and compact summary.
from IPython.display import Image as NotebookImage, Markdown, display

assert FIGURE_PATH.is_file() and FIGURE_PATH.stat().st_size > 0, FIGURE_PATH
display(Markdown(f"## Figure 5: `{FIGURE_PATH.relative_to(ROOT)}`"))
display(NotebookImage(filename=str(FIGURE_PATH)))
display(Markdown("## Fixed-split Figure 5 summary"))
display(pd.read_csv(summary_path))
